# Deep Learning 078 — Positional Encoding

Companion notebook to the lesson. Self-attention processes all words at once, which is what
makes it fast — and it is exactly why it cannot tell "nitish killed lion" from "lion killed
nitish". This notebook derives the fix from first principles, testing each attempt and
watching it fail before moving to the next.

| Step | What we measure |
|---|---|
| the problem | reversing the sentence changes the output by **6.7e-16** |
| counting | a book reaches position **99,999** |
| normalising by length | position 2 is **1.000** in a 2-word sentence, **0.500** in a 4-word one |
| one sine | positions **11 and 344** land 1.3e-07 apart |
| sin/cos pairs, falling frequency | at d=128 the closest pair is **4885 and 4886 — adjacent** |
| the relative-position property | `PE(pos+k) = M_k PE(pos)` to **4.6e-14**, and `M_k` is a rotation |
| why cosine, not just sine | the best sine-only linear map is off by **1.17** |

`numpy` only; the heat-map cell uses `matplotlib` and can be skipped.

In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)

## Part A — The problem is an identity, not a weakness

"Self-attention cannot capture word order" sounds like something training could fix. It is
stronger than that: self-attention is **permutation-equivariant**. Permute the input rows
and the output rows permute with them, otherwise unchanged.

In [ ]:
d = 32
Wq, Wk, Wv = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))

def self_attention(X):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    return softmax(Q @ K.T / np.sqrt(d)) @ V

E = {w: rng.normal(size=d) for w in ("nitish", "killed", "lion")}
X1 = np.stack([E[w] for w in ("nitish", "killed", "lion")])
X2 = np.stack([E[w] for w in ("lion", "killed", "nitish")])

Y1, Y2 = self_attention(X1), self_attention(X2)
print(f"max |Y2 - reverse(Y1)|         = {np.abs(Y2 - Y1[[2, 1, 0]]).max():.1e}")
print(f"max |sum(Y1) - sum(Y2)|        = {np.abs(Y1.sum(0) - Y2.sum(0)).max():.1e}")
assert np.abs(Y2 - Y1[[2, 1, 0]]).max() < 1e-14

In [ ]:
# for contrast: a recurrent encoder reads one word at a time, so order is in the computation
Wh = rng.normal(size=(d, d)) / np.sqrt(d)
Ux = rng.normal(size=(d, d)) / np.sqrt(d)

def rnn(X):
    h = np.zeros(d)
    for x in X:                                  # this loop is the whole difference
        h = np.tanh(h @ Wh + x @ Ux)
    return h

print(f"max |rnn(sentence) - rnn(reversed)| = {np.abs(rnn(X1) - rnn(X2)).max():.4f}")

The two sentences do not produce *similar* outputs — they produce the **same vectors in a
different order**, so any order-blind operation on top (a sum, a mean, a pooling layer) sees
literally identical input. The `1e-16` residue is float64 addition not being associative.

**No gradient can fix this, because there is no parameter it depends on.** The order
information has to be put into the vectors before they reach the block. That is all
positional encoding is.

## Part B — Attempt 1: just count

Append the position as one extra number. It fails three tests.

In [ ]:
print("unbounded:")
for n, label in ((4, "a 4-word sentence"), (100_000, "a 500-page book")):
    print(f"   {label:<20} last position = {n - 1:>7,}")
print("   backprop wants values roughly in [-1, 1]\n")

print("normalising by sentence length fixes the range and breaks something worse:")
print(f"   {'sentence length':>17}{'value at position 2':>22}")
for n in (2, 4, 10, 50):
    print(f"   {n:>17}{2 / n:>22.4f}")

Unboundedness is a nuisance you could rescale away. **Inconsistency is fatal.** A positional
code has to mean the same thing in every training example — if position 2 is 1.000 in one
sentence and 0.040 in another, the model is being taught a different language every batch.

So the requirements are now explicit: **bounded**, **consistent**, and **continuous** (so
"two apart" is a computable thing). Bounded plus continuous plus defined everywhere points
at trigonometric functions.

## Part C — Attempt 2: one sine, and how badly it repeats

`sin(pos)` is bounded, continuous and consistent. It also repeats. The useful question is
not *whether* that matters but *how much*, so measure the closest any two positions get.

In [ ]:
def positional_encoding(n_pos, d_model, base=10000.0):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angle = pos / base ** (2 * i / d_model)     # frequency falls as i grows
    pe = np.zeros((n_pos, d_model))
    pe[:, 0::2] = np.sin(angle)                 # even dimensions: sine
    pe[:, 1::2] = np.cos(angle)                 # odd dimensions: cosine
    return pe

def min_gap(mat):
    sq = (mat ** 2).sum(1)
    d2 = sq[:, None] + sq[None, :] - 2 * mat @ mat.T
    np.fill_diagonal(d2, np.inf)
    return np.sqrt(max(d2.min(), 0)), np.unravel_index(d2.argmin(), d2.shape)

N = 5000
p = np.arange(N)
variants = [("sin(pos) alone", np.sin(p)[:, None]),
            ("sin(pos), cos(pos)", np.stack([np.sin(p), np.cos(p)], 1)),
            ("2 pairs  (d = 4)", positional_encoding(N, 4)),
            ("8 pairs  (d = 16)", positional_encoding(N, 16)),
            ("64 pairs (d = 128)", positional_encoding(N, 128))]

print(f"positions 0-{N - 1}, exact minimum distance between any two encodings:\n")
print(f"{'encoding':<22}{'min distance':>14}   closest pair")
for name, mat in variants:
    gap, (a, b) = min_gap(mat)
    print(f"{name:<22}{gap:>14.2e}   ({a}, {b})")

**Read the last column, not the middle one.**

A single sine puts positions 11 and 344 on top of each other — the model would be told they
are the same place. The sin/cos pair is better and still not safe. But from `d = 16` upward
the closest pair becomes **adjacent**: there is no distant collision left anywhere in the
first 5,000 positions, so the only positions that resemble each other are the ones that
should. That is a much stronger statement than "collisions become unlikely".

## Part D — The formula, and computing it by hand once

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
print("d_model = 6, first two words:\n")
print(positional_encoding(2, 6).round(4))
print("\nposition 0 is exactly [0, 1, 0, 1, 0, 1] - every angle is 0, so sin=0 and cos=1")
print("that alternating stripe is the top row of every PE heat map you have seen")

In [ ]:
# the wavelengths form a geometric progression - exactly like binary bit periods
D_MODEL = 128
i = np.arange(D_MODEL // 2)
wavelength = 2 * np.pi * 10000.0 ** (2 * i / D_MODEL)

print("binary: bit k has period 2^k  ->  1, 2, 4, 8, ...   ratio 2.0000\n")
print(f"{'pair i':>8}{'wavelength':>14}{'ratio to previous':>20}")
for j in (0, 1, 2, 8, 32, 63):
    r = "" if j == 0 else f"{wavelength[j] / wavelength[j - 1]:>20.4f}"
    print(f"{j:>8}{wavelength[j]:>14,.1f}{r}")
print(f"\nconstant ratio {wavelength[1] / wavelength[0]:.4f} = 10000^(2/{D_MODEL})")
print(f"longest wavelength {wavelength[-1]:,.0f} positions")

**Positional encoding is binary counting rendered in continuous numbers.** Binary doubles
the period at every bit; this multiplies it by 1.1548 at every pair. Same geometric
structure, gentler steps, and continuous values because gradients need somewhere to flow.

The longest wavelength is 54,410 positions — the design deciding in advance how long a
document it intends to index. Which explains the shape of the famous heat map:

In [ ]:
pe50 = positional_encoding(50, D_MODEL)
print(f"{'dimension':>11}{'range over positions 0-49':>28}")
for col in (0, 1, 20, 60, 100, 126):
    print(f"{col:>11}{pe50[:, col].max() - pe50[:, col].min():>28.6f}")

In [ ]:
# Optional heat map. Skip this cell if matplotlib is not installed.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(positional_encoding(50, 128), aspect="auto", cmap="RdBu")
ax[0].set(xlabel="dimension", ylabel="position", title="50 positions, d_model = 128")
fig.colorbar(im, ax=ax[0])

for dim, lab in ((0, "dim 0, wavelength 6.3"), (20, "dim 20, wavelength 26.5"),
                 (40, "dim 40, wavelength 111.7"), (60, "dim 60, wavelength 471")):
    curve = positional_encoding(60, 128)[:, dim]
    ax[1].plot(curve, label=lab)
ax[1].set(xlabel="position", ylabel="value", title="each pair slower than the last")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

The low dimensions swing across the full range and do all the work of separating nearby
words. Dimension 126 moves by 0.0057 across the whole sentence and separates nothing —
those columns are for position 3,000 versus position 4,000 in a long document.

## Part E — The part that justifies the cosines

Everything so far would work with sines at many frequencies and no cosines. The pairing is
usually waved through as "it helps uniqueness". The real reason is sharper.

Take one sin/cos pair at frequency ω. Its two entries are `(sin ωpos, cos ωpos)` — **a point
on a circle at angle ωpos**. Moving to `pos+k` turns it by `ωk`, and a turn is a 2×2
rotation that depends on `k` and `ω` but **not on `pos`**. Stack one block per pair and you
get a single matrix `M_k` that advances *every* position at once.

In [ ]:
D = 64
PE = positional_encoding(600, D)

def shift_matrix(k, d_model, base=10000.0):
    M = np.zeros((d_model, d_model))
    for j in range(d_model // 2):
        w = 1.0 / base ** (2 * j / d_model)
        c, s = np.cos(k * w), np.sin(k * w)
        M[2*j, 2*j],     M[2*j, 2*j+1]     = c, s
        M[2*j+1, 2*j],   M[2*j+1, 2*j+1]   = -s, c
    return M

worst = 0.0
for k in (1, 2, 5, 10, 37, 100):
    M = shift_matrix(k, D)
    err = np.abs(PE[k:k + 400] - PE[:400] @ M.T).max()
    worst = max(worst, err)
    print(f"offset k = {k:>3}: one matrix maps PE(pos) -> PE(pos+{k}) for all 400 "
          f"positions, max error {err:.1e}")
assert worst < 1e-12

In [ ]:
# the control: can a sine-only code do the same? Find the BEST linear map by least squares.
sin_only = np.sin(np.arange(600)[:, None] / 10000.0 ** (2 * np.arange(D) / D))
best, *_ = np.linalg.lstsq(sin_only[:400], sin_only[10:410], rcond=None)
resid = np.abs(sin_only[10:410] - sin_only[:400] @ best).max()

print(f"sine only,   k = 10, best possible linear map: max error {resid:.2e}")
print(f"sin/cos,     k = 10, the rotation above      : max error "
      f"{np.abs(PE[10:410] - PE[:400] @ shift_matrix(10, D).T).max():.2e}")

**That is the answer to "why cosine as well".** With pairs, the shift is exact. With sines
alone the best linear map that *exists* — found by least squares, not guessed — is wrong by
1.17 on a scale of [-1, 1]. A rotation needs both coordinates: given only `sin ωpos` you
cannot tell whether the circle is on its way up or down, and no matrix recovers what was
never encoded.

The cosine is not there to reduce collisions. **It is the other half of the rotation.**

And the consequence is the property that actually matters, because attention scores *are*
dot products:

In [ ]:
print(f"{'offset k':>10}{'mean PE(p).PE(p+k)':>22}{'std across p':>16}")
for k in (0, 1, 2, 5, 10, 50, 200):
    dots = np.einsum("ij,ij->i", PE[:300], PE[k:k + 300])
    print(f"{k:>10}{dots.mean():>22.4f}{dots.std():>16.2e}")

The right-hand column is the result: **the similarity between two positions has no
dependence on where you are, only on how far apart you are.** Absolute position went in,
relative position came out.

## Part F — Added, not concatenated

Two reasons, and the standard one understates the case.

In [ ]:
D_M = 512
print("attention parameters are 3 * d^2:\n")
print(f"   {'adding (d stays 512)':<28}{3 * D_M * D_M:>12,}")
print(f"   {'concatenating (d -> 1024)':<28}{3 * (2 * D_M) ** 2:>12,}")
print(f"\n   that is {3 * (2 * D_M) ** 2 / (3 * D_M * D_M):.0f}x, not 2x - the matrices are d BY d")

In [ ]:
# the reason nobody mentions: relative magnitude decides what survives an addition
print("half the entries are sines and half cosines, and sin^2 + cos^2 = 1 per pair:\n")
for dm in (6, 128, 512):
    p_ = positional_encoding(200, dm)
    print(f"   d_model {dm:>4}: mean ||PE|| = {np.linalg.norm(p_, axis=1).mean():>7.4f}"
          f"   predicted sqrt(d/2) = {np.sqrt(dm / 2):.4f}")

emb = rng.normal(size=(20, 512))
emb /= np.linalg.norm(emb, axis=1, keepdims=True)       # unit-norm word embeddings
pe = positional_encoding(20, 512)
print()
for scale, label in ((1.0, "unscaled embedding"), (np.sqrt(512), "scaled by sqrt(d_model)")):
    share = np.linalg.norm(emb * scale, axis=1) / (
        np.linalg.norm(emb * scale, axis=1) + np.linalg.norm(pe, axis=1))
    print(f"   {label:<26} the word carries {share.mean():.1%} of the summed length")

`||PE|| = sqrt(d/2) = 16` at `d_model = 512`. Add that to a unit-norm word embedding and the
position contributes 94% of the length — **the word is nearly erased.**

This is why the paper multiplies the embeddings by `sqrt(d_model) = 22.6` before the
addition, a line that is almost always skipped in explanations. Measured, the word's share
goes from **5.9%** to **58.6%**. Without it, "adding position to meaning" is mostly just
position.

## Summary

Self-attention is permutation-equivariant, so order has to arrive inside the vectors.
Positional encoding supplies it with a bounded, continuous, consistent code: sin/cos pairs
at geometrically falling frequencies, `d_model`-dimensional, added to the scaled embedding.
It is binary counting in continuous numbers, it has **zero learned parameters**, and its
sin/cos structure makes advancing `k` positions an exact rotation — which is what turns
absolute position into the relative position dot products can use.

## Try it yourself

1. Change the base from 10000 to 100 and re-run Part C. What happens to the minimum gap over
   5,000 positions, and why?
2. Build a *learned* positional embedding (a `n_pos × d_model` array of free parameters).
   What does it lose that the sinusoidal one has? (Test `PE(pos+k) = M_k PE(pos)` on it.)
3. Verify `||PE|| = sqrt(d/2)` symbolically for one pair, then explain why it holds exactly
   rather than on average.
4. Feed `X + PE` into the Part A attention and confirm the reversed sentence now gives a
   genuinely different output. How different?